# İP5 — Random Forest + Adaptif AKTS

**Görev (Bölüm 10, Görev 6):** Adaptif AKTS Hesaplayan Tahmin Modeli Oluşturma
**Sorumlu:** Samet Koca | **Tarih:** 18 Temmuz 2026

**Girdi:** `data/processed/model_egitim.xlsx` (43 sütun) + `data/processed/master_bloom.xlsx` (K_profil, K_etkinlik)

**Çıktı:** `data/processed/master_akts_final.xlsx` — her ders için `rf_tahmin_akts`, `kural_akts`, `adaptif_akts`, `sapma`, `karar`

Bu notebook, README.md Bölüm 14-15'te tartışılan altı kararı uygulayan `ip5-random-forest/src/ip5_random_forest.py` pipeline'ını çalıştırır ve sonuçları görselleştirir.

### Formüller

$$kural\_akts = akts\_mevcut \times K\_etkinlik \times K\_profil \times K\_bilissel$$

$$adaptif\_akts = 0.70 \times rf\_tahmin\_akts + 0.30 \times kural\_akts$$

**Hedef sızıntısı önlemi:** `akts_mevcut` (AKTSKredi) ve `K_bilissel`/`K_profil`/`K_etkinlik` RF'in özellik kümesine DAHİL EDİLMEZ — sadece kural tabanlı bileşende kullanılır (Karar 2b).

In [1]:
import sys
from pathlib import Path

BASE = Path.cwd().resolve().parents[1]  # course-credits-system köküne çık
sys.path.insert(0, str(BASE / "ip5-random-forest" / "src"))

import ip5_random_forest as ip5

final_df = ip5.main()

İP5 — RANDOM FOREST + ADAPTİF AKTS

Yükleniyor: model_egitim.xlsx


  4133 satır, 43 sütun
Yükleniyor: master_bloom.xlsx


  4133 satır, 107 sütun

Özellik sayısı (X): 39
Hariç tutulan sütunlar: ['Katalogid', 'DersAdi', 'AKTSKredi', 'K_bilissel']



── Model Kalitesi (ayrılmış %20 test seti) ──────────────
  MAE  : 0.8352 AKTS
  RMSE : 1.1544 AKTS
  R²   : 0.0781

5 katlı stratified (Fakülte) çapraz doğrulama ile out-of-fold tahmin üretiliyor...


  Fold 1/5 tamamlandı (827 ders tahmin edildi)
  Fold 2/5 tamamlandı (827 ders tahmin edildi)


  Fold 3/5 tamamlandı (827 ders tahmin edildi)


  Fold 4/5 tamamlandı (826 ders tahmin edildi)


  Fold 5/5 tamamlandı (826 ders tahmin edildi)

── Model Kalitesi (tüm veri, out-of-fold) ────────────────
  MAE  : 0.9285 AKTS
  RMSE : 1.7504 AKTS
  R²   : -0.1164



── En Önemli 10 Özellik (feature_importances_) ───────────
  bloom_avg_level             : 0.2610
  bloom_max_level             : 0.1448
  n_ogrenme_kazanim           : 0.1030
  n_etkinlik                  : 0.0710
  has_odev                    : 0.0386
  Fakulte_Eğitim              : 0.0338
  Fakulte_Güzel Sanatlar      : 0.0271
  derstur_Teorik              : 0.0268
  alan_grubu_Sanat            : 0.0252
  alan_grubu_Spor             : 0.0249

── Karar Dağılımı (0.70/0.30 ağırlık, primary) ───────────
  Güçlü uyum        :  1872 ders  (45.3%)
  Kabul edilebilir  :  1623 ders  (39.3%)
  İncelenmeli       :   638 ders  (15.4%)

── Duyarlılık Analizi: Ağırlık Şemalarına Göre Ortalama |Sapma%| ──
  0.50/0.50  -> ort |sapma%|=14.72  incelenmeli oranı=11.32%
  0.60/0.40  -> ort |sapma%|=16.11  incelenmeli oranı=12.94%
  0.70/0.30  -> ort |sapma%|=17.74  incelenmeli oranı=15.44%
  0.80/0.20  -> ort |sapma%|=19.59  incelenmeli oranı=18.00%



Kaydedildi: /Users/sudenazguven/Desktop/akts/data/processed/master_akts_final.xlsx
Kaydedildi: /Users/sudenazguven/Desktop/akts/ip5-random-forest/feature_importance.xlsx
Kaydedildi: /Users/sudenazguven/Desktop/akts/ip5-random-forest/duyarlilik_analizi.xlsx
Kaydedildi: /Users/sudenazguven/Desktop/akts/ip5-random-forest/IP5_SONUC_RAPORU.md


### Sonuçları İncele

In [2]:
final_df[['DersAdi', 'Fakulte', 'AKTSKredi', 'rf_tahmin_akts', 'kural_akts', 'adaptif_akts', 'sapma_yuzde', 'karar']].sample(10, random_state=1)

,DersAdi,Fakulte,AKTSKredi,rf_tahmin_akts,kural_akts,adaptif_akts,sapma_yuzde,karar
2534,Hidrolik Makinalar,Mühendislik,6,4.2997,5.9322,4.7894,-20.18,Kabul edilebilir
804,İleri Laboratuvar II,Fen-Edebiyat,5,5.0505,8.4139,6.0595,21.19,Kabul edilebilir
3486,Medikal İllüstrasyon II,Güzel Sanatlar,4,3.3394,4.2712,3.6189,-9.53,Güçlü uyum
709,Elektrokimya,Fen-Edebiyat,5,4.2546,5.1907,4.5354,-9.29,Güçlü uyum
2950,Ayrık Matematik,Mühendislik,4,3.6787,4.3503,3.8802,-3.00,Güçlü uyum
1690,Doğalgaz ve Sıhhi Tesisat,Teknoloji,4,4.1895,4.1003,4.1627,4.07,Güçlü uyum
829,Bitirme Çalışması,Fen-Edebiyat,1,2.6300,1.3100,2.2340,123.40,İncelenmeli
3613,Toplu Müzik Eğitimi 5,Güzel Sanatlar,3,2.6004,2.9661,2.7101,-9.66,Güçlü uyum
1328,EĞLENCELİ ATLETİZM,Spor Bilimleri,4,4.4200,4.6327,4.4838,12.10,Kabul edilebilir
2610,Çözelti Termodinamiği,Mühendislik,4,4.1184,3.9548,4.0693,1.73,Güçlü uyum


In [3]:
import pandas as pd

importance_df = pd.read_excel(BASE / 'ip5-random-forest' / 'feature_importance.xlsx')
importance_df.head(15)

,ozellik,onem_derecesi
0,bloom_avg_level,0.260957
1,bloom_max_level,0.144849
2,n_ogrenme_kazanim,0.102967
3,n_etkinlik,0.070965
4,has_odev,0.038634
5,Fakulte_Eğitim,0.033834
6,Fakulte_Güzel Sanatlar,0.027088
7,derstur_Teorik,0.026801
8,alan_grubu_Sanat,0.025189
9,alan_grubu_Spor,0.024917


In [4]:
sensitivity_df = pd.read_excel(BASE / 'ip5-random-forest' / 'duyarlilik_analizi.xlsx')
sensitivity_df

,agirlik,ort_mutlak_sapma_yuzde,incelenmeli_oran_yuzde
0,0.50 RF / 0.50 Kural,14.72,11.32
1,0.60 RF / 0.40 Kural,16.11,12.94
2,0.70 RF / 0.30 Kural,17.74,15.44
3,0.80 RF / 0.20 Kural,19.59,18.00


### Bulgu ve İP6'ya Devir

Model kalitesi (out-of-fold) düşük/negatif R² gösteriyor — bu beklenen bir durumdur: amaç mevcut AKTS'yi ezberlemek değil, ondan anlamlı biçimde sapan dersleri işaretlemektir ("iyi taklit paradoksu", README.md §15.5). Ayrıntılı rapor: `ip5-random-forest/IP5_SONUC_RAPORU.md`.

Çıktı (`master_akts_final.xlsx`), İP6'da (Görev 8) tutarsızlık analizi ve hipotez testleri için girdi olarak kullanılır.